[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_38_QLoRA_Fine_Tuning_on_Real_Data.ipynb)

# Lesson 38 — QLoRA Fine-Tuning on Real Data
**Track 3 · Self-hosted & Fine-tuning | Lesson 2 of 5**

## What We'll Build
By the end of this notebook you will have:
1. Fine-tuned `Qwen2.5-0.5B-Instruct` on **SQL generation** using **QLoRA** (~5-8 min on a free T4)
2. Evaluated quality with an **LLM-as-judge** harness (the L17 pattern)
3. **Merged** LoRA adapter weights back into the base model
4. **Served the merged model with vLLM** — closing the L37 → L38 loop
5. Built intuition for *when* fine-tuning beats prompting/RAG

## Track 3 Roadmap
| Lesson | Topic | Status |
|--------|-------|--------|
| L37 | vLLM: Serve Your Own LLM | ✅ Done |
| **L38** | **QLoRA Fine-tuning on Real Data** | ⬅ You are here |
| L39 | DPO/ORPO: Preference Tuning | ⏳ |
| L40 | Knowledge Distillation | ⏳ |
| L41 | Model Merging (SLERP/TIES/DARE) | ⏳ |

> ⚡ **Before you start**: Runtime → Change runtime type → **T4 GPU**
> This notebook requires a GPU. A free T4 (15 GB VRAM) is sufficient.

## 1. When Does Fine-Tuning Win?

Fine-tuning is **not always the right tool**. Use this decision tree first:

```
Is your task well-defined with consistent input/output structure?
├─ No  → Use prompting (L2) or RAG (L7)
└─ Yes: Do you have 50–10,000 quality labeled examples?
         ├─ No  → Few-shot prompting or RAG
         └─ Yes: Is one of these true?
                  ├─ Format must be consistent (JSON, SQL, specific style)
                  ├─ Latency-critical (long system prompts are too slow)
                  ├─ Privacy — data cannot leave your servers
                  └─ Volume > 5M tokens/day (self-hosted cost math wins)
                  → Yes to any: Fine-tune ✅
```

### Cost Math — Fine-tuned 1.5B vs General Models (100M tokens/month)
| Approach | Cost/1M tok | Setup Cost | Quality on Narrow Task |
|---|---|---|---|
| Claude Haiku API | $0.80 | $0 | Good (general) |
| Self-hosted 7B vLLM (L37) | ~$0.01 | $0 | Good (general) |
| **Fine-tuned 1.5B vLLM (this lesson)** | **~$0.005** | **~$5 one-time** | **Best (narrow)** |

**Key insight**: A well fine-tuned 1.5B *beats* a general 7B on the specific task it was trained for — at 5× lower serving cost.

### Why SQL Generation?
- Clear, measurable output (SQL is either correct or wrong)
- Consistent format required (small models struggle without training)
- Realistic use case (every company wants natural language → SQL)
- Fast to evaluate (LLM judge can assess SQL correctness in one call)

In [ ]:
# ── Setup — takes ~2 min first time ──────────────────────────────────────────
!pip install -q \
    "transformers>=4.40.0" \
    "peft>=0.10.0" \
    "trl>=0.8.6" \
    "bitsandbytes>=0.43.0" \
    "datasets>=2.18.0" \
    "accelerate>=0.28.0" \
    "anthropic" \
    "sentencepiece"

print("✅ Libraries installed")

In [ ]:
import os, json, time, gc, statistics
from dataclasses import dataclass

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer
import anthropic

# ── Anthropic API key ─────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    print("⚠️  Set manually: os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'")

# ── GPU check ─────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"✅ GPU  : {props.name}")
    print(f"   VRAM : {vram_gb:.1f} GB")
    if vram_gb < 12:
        print("⚠️  < 12 GB — will use 4-bit quantization (already planned)")
else:
    raise RuntimeError("❌ No GPU found. Go to Runtime → Change runtime type → T4 GPU")

## 2. QLoRA Architecture — The Math

### LoRA (Low-Rank Adaptation)
Standard fine-tuning updates all weights: `W_new = W_orig + ΔW`
For a 1.5B model that's ~3 GB of trainable gradients — too large for a free GPU.

LoRA approximates ΔW with two tiny matrices:
```
ΔW = A × B     where A ∈ R^(d × r),  B ∈ R^(r × k)
```
With rank `r = 16` and d = k = 2048:
- Full ΔW: 2048² = **4,194,304 params**
- LoRA ΔW: 2 × (2048 × 16) = **65,536 params** ← 64× smaller

### QLoRA Adds 4-bit Quantization of the Base Weights
```
QLoRA = frozen 4-bit base weights  +  trainable BF16 LoRA adapters

Base model: INT4 (NF4 format) ← frozen, loaded in 4-bit, never updated
Adapters:   BF16              ← tiny, trainable (<100 MB for 1.5B model)
```
Gradients only flow through the adapters. The frozen base weights just do
the forward pass (dequantized to BF16 on the fly — "double quantization").

Reference: Dettmers et al. 2023 — "QLoRA: Efficient Finetuning of Quantized LLMs"

### Key Hyperparameters
| Param | Typical | Controls |
|-------|---------|----------|
| `r` (rank) | 8–64 | Adapter expressiveness. Higher r = more params |
| `lora_alpha` | 16–64 | Scale = alpha/r. Rule of thumb: alpha = 2×r |
| `lora_dropout` | 0.05 | Regularization on adapter weights |
| `target_modules` | q/k/v/o_proj | Which linear layers to adapt |
| `learning_rate` | 2e-4 | LoRA LR is 10× higher than full fine-tuning |

In [ ]:
# ── VRAM budget: training vs inference ───────────────────────────────────────
# Inference  (L37): weights + KV cache
# Training adds   : gradients + optimizer states + activations
#
# With gradient checkpointing (recompute activations instead of storing them):
#   Qwen2.5-0.5B INT4 training ≈ 0.4 + 0.2 + 0.5 GB overhead ≈ ~2 GB total
#   Qwen2.5-1.5B INT4 training ≈ 1.2 + 0.5 + 1.0 GB overhead ≈ ~3.5 GB total
#   Qwen2.5-7B   INT4 training ≈ 4.5 + 1.5 + 3.0 GB overhead ≈ ~9 GB total

@dataclass
class TrainEstimate:
    name: str
    params_b: float
    dtype: str            # INT4 | BF16

    @property
    def base_gb(self):
        bpp = {"INT4": 0.5, "INT8": 1.0, "BF16": 2.0}[self.dtype]
        return self.params_b * 1e9 * bpp / 1e9

    @property
    def adapter_gb(self):
        return self.params_b * 0.003 * 2   # ~0.3% params in BF16

    @property
    def overhead_gb(self):
        return max(0.5, self.params_b * 0.15)  # activations + optimizer (w/ grad ckpt)

    @property
    def total_gb(self):
        return self.base_gb + self.adapter_gb + self.overhead_gb

models = [
    TrainEstimate("Qwen2.5-0.5B", 0.5,  "INT4"),
    TrainEstimate("Qwen2.5-1.5B", 1.5,  "INT4"),
    TrainEstimate("Qwen2.5-7B",   7.0,  "INT4"),
    TrainEstimate("Llama-3.1-8B", 8.0,  "INT4"),
    TrainEstimate("Mistral-7B",   7.0,  "INT4"),
]

print(f"{'Model':<22} {'Base':>7} {'Adapter':>8} {'Ovhd':>7} {'Total':>7}  T4 (15 GB)")
print("─" * 70)
for m in models:
    fit = "✅ Fits" if m.total_gb < 12 else "⚠️ Tight" if m.total_gb < 15 else "❌ OOM"
    print(f"{m.name:<22} {m.base_gb:>6.1f}G {m.adapter_gb:>7.1f}G {m.overhead_gb:>6.1f}G {m.total_gb:>6.1f}G  {fit}")

print()
print("→ Lesson uses Qwen2.5-0.5B INT4: fastest training, still shows the concept")
print("  For production quality: use 1.5B or 7B INT4")

## 3. Dataset — What Makes Good Fine-Tuning Data

### The Standard SFT Format
Most supervised fine-tuning datasets follow an **instruction-input-output** structure.
For SQL generation it looks like:
```
User:      Given this schema: CREATE TABLE employees(id, name, salary)
           Question: What is the average salary?
Assistant: SELECT AVG(salary) FROM employees
```

### Data Quality Rules
1. **Correctness over size** — 100 high-quality > 1,000 noisy examples
2. **Consistency** — every example must use the exact same format you want at inference
3. **Diversity** — cover the full input distribution (simple SELECTs + JOINs + GROUP BY)
4. **No leakage** — hold out a test split BEFORE you start; never eval on train data

### Our Dataset: `b-mc2/sql-create-context`
- ~78K (schema + NL question) → SQL answer pairs
- We'll use **200 train + 50 test** (sufficient for a fast demo)
- A production QLoRA run would use 1,000–5,000 examples

In [ ]:
# ── Load & inspect the dataset ───────────────────────────────────────────────
print("Downloading sql-create-context ...")
raw = load_dataset("b-mc2/sql-create-context", split="train")
print(f"Full dataset: {len(raw):,} examples")

sample = raw[42]
print("\nSample record:")
for k, v in sample.items():
    print(f"  {k:12s}: {str(v)[:120]}{'...' if len(str(v)) > 120 else ''}")

# ── Subsample ─────────────────────────────────────────────────────────────────
# 💡 EXPERIMENT: increase TRAIN_SIZE to 500-1000 for better quality
TRAIN_SIZE = 200
TEST_SIZE  = 50
SEED       = 42

raw_shuffled = raw.shuffle(seed=SEED)
train_raw = raw_shuffled.select(range(TRAIN_SIZE))
test_raw  = raw_shuffled.select(range(TRAIN_SIZE, TRAIN_SIZE + TEST_SIZE))

print(f"\n✅ Train: {len(train_raw)} | Test: {len(test_raw)}")

In [ ]:
# ── Load tokenizer + format into chat template ───────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# 💡 EXPERIMENT: try "Qwen/Qwen2.5-1.5B-Instruct" for better quality (takes ~10 min)

print(f"Loading tokenizer for {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM_PROMPT = (
    "You are a SQL expert. Given a table schema and a natural language question, "
    "write a valid SQL query that answers the question. Return ONLY the SQL query, "
    "no explanation."
)

def format_as_chat(example):
    user_msg = (
        f"Schema:\n{example['context']}\n\n"
        f"Question: {example['question']}"
    )
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": example["answer"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_ds = train_raw.map(format_as_chat, remove_columns=train_raw.column_names)
test_ds  = test_raw.map(format_as_chat,  remove_columns=test_raw.column_names)

print("\nFormatted example (first 500 chars):")
print(train_ds[0]["text"][:500])
print("...")

# Check token lengths
lengths = [len(tokenizer.encode(ex["text"])) for ex in train_ds]
p95 = sorted(lengths)[int(0.95 * len(lengths))]
print(f"\nToken lengths — mean: {statistics.mean(lengths):.0f}, max: {max(lengths)}, p95: {p95}")
print(f"→ max_seq_length=512 covers p95 comfortably")

print(f"\n✅ Train: {len(train_ds)} formatted | Test: {len(test_ds)} formatted")

## 4. Load Model in 4-bit + Apply LoRA

Three steps to set up QLoRA:

**Step 1** — `BitsAndBytesConfig`: tells PyTorch to load weights in **NF4** (4-bit NormalFloat), the best quantization format for LLM weight distributions.

**Step 2** — `AutoModelForCausalLM.from_pretrained(..., quantization_config=...)`: loads the base model with frozen 4-bit weights.

**Step 3** — `get_peft_model(model, lora_config)`: injects tiny A/B adapter matrices into the attention projection layers. After this, `requires_grad=True` only for the adapters (~1–2% of all parameters).

The base model is **completely frozen** — backprop never touches it.

In [ ]:
# ── 4-bit quantization config (NF4 + double quantization) ────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",               # NormalFloat4 — best for LLM weights
    bnb_4bit_compute_dtype=torch.bfloat16,   # BF16 compute during forward pass
    bnb_4bit_use_double_quant=True,          # quantize the quant constants (saves ~0.4 GB)
)

print(f"Loading {MODEL_NAME} in 4-bit NF4 ...")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",          # auto-shards across available GPUs
    trust_remote_code=True,
)
print(f"✅ Loaded in {time.time()-t0:.1f}s  |  Footprint: {model.get_memory_footprint()/1e9:.2f} GB")

# ── LoRA config ───────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,                     # rank
    lora_alpha=32,            # scale = alpha/r = 2
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~1% trainable (adapters only) — base weights are frozen

# Needed for gradient checkpointing with quantized models
model.enable_input_require_grads()
model.gradient_checkpointing_enable()
print("\n✅ QLoRA model ready")

## 5. Train with SFTTrainer

`SFTTrainer` from the `trl` library wraps HuggingFace's `Trainer` with:
- **Packing** (`packing=True`): bins multiple short examples into one `max_seq_length` window → ~2× throughput
- **Dataset collation**: handles padding and attention masks automatically
- **Loss masking**: can mask out the prompt tokens so loss is computed only on the assistant's output

### Key Training Decisions
| Param | Value | Why |
|-------|-------|-----|
| `num_train_epochs` | 3 | Enough to learn the format; more risks overfitting 200 examples |
| `per_device_train_batch_size` | 4 | T4 handles this with 4-bit + grad checkpointing |
| `gradient_accumulation_steps` | 2 | Effective batch = 4×2 = 8 |
| `learning_rate` | 2e-4 | 10× higher than full fine-tuning (LoRA adapters start at 0) |
| `optim` | paged_adamw_32bit | Keeps optimizer states on CPU — saves ~1 GB VRAM |
| `packing` | True | Packs multiple short SQL examples per 512-token window |

### Watch the Loss
- Start: ~2.5–3.0 (random adapter weights)
- After 1 epoch: ~1.0–1.5
- After 3 epochs: ~0.4–0.8 on this small dataset
- If loss plateaus above 1.5, try a higher LR or more data

In [ ]:
OUTPUT_DIR  = "/content/qlora_sql"
ADAPTER_DIR = f"{OUTPUT_DIR}/final_adapter"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,    # effective batch = 8
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",                 # disable wandb
    optim="paged_adamw_32bit",        # optimizer states on CPU
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=512,
    packing=True,
    args=training_args,
)

steps_estimate = len(trainer.get_train_dataloader()) * training_args.num_train_epochs
print(f"Training {TRAIN_SIZE} examples for {training_args.num_train_epochs} epochs")
print(f"Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Estimated total steps: {steps_estimate}")
print("\nStarting ... (watch loss drop in the log below)")
print("─" * 50)

t_start = time.time()
trainer.train()
t_train = time.time() - t_start
print(f"\n✅ Training complete in {t_train/60:.1f} min")

trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ Adapter saved to {ADAPTER_DIR}")

## 6. Before vs After: Did It Work?

The simplest sanity check: run the same SQL questions through the **fine-tuned model** and compare to what a base 0.5B would produce.

We expect the fine-tuned model to:
- Generate cleaner, more consistent SQL
- Follow the schema more precisely
- Avoid hallucinating column names

*Note: 200 training examples is a minimal demo. A production fine-tune uses 1K–5K examples and shows much larger quality gains.*

In [ ]:
# ── SQL generation helper ─────────────────────────────────────────────────────
def generate_sql(m, tok, schema: str, question: str, max_new_tokens=150) -> str:
    user_msg = f"Schema:\n{schema}\n\nQuestion: {question}"
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_msg},
    ]
    prompt = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tok(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out_ids = m.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tok.eos_token_id,
        )
    new_ids = out_ids[0, inputs["input_ids"].shape[1]:]
    return tok.decode(new_ids, skip_special_tokens=True).strip()

# ── Test cases ────────────────────────────────────────────────────────────────
test_cases = [
    {
        "schema": "CREATE TABLE orders (order_id INT, customer_id INT, total DECIMAL, status VARCHAR, created_at DATE)",
        "question": "Find all pending orders with total > 100, sorted by creation date descending.",
        "expected": "SELECT * FROM orders WHERE total > 100 AND status = 'pending' ORDER BY created_at DESC",
    },
    {
        "schema": "CREATE TABLE employees (emp_id INT, name VARCHAR, department VARCHAR, salary DECIMAL)",
        "question": "What is the average salary per department, only for departments with more than 5 employees?",
        "expected": "SELECT department, AVG(salary) FROM employees GROUP BY department HAVING COUNT(*) > 5",
    },
    {
        "schema": "CREATE TABLE products (product_id INT, name VARCHAR, price DECIMAL, category VARCHAR, stock INT)",
        "question": "List the names and prices of the 5 most expensive products.",
        "expected": "SELECT name, price FROM products ORDER BY price DESC LIMIT 5",
    },
]

print("Fine-tuned model output on held-out test cases:")
print("=" * 70)
for i, tc in enumerate(test_cases):
    print(f"\nTest {i+1}: {tc['question']}")
    print(f"Expected : {tc['expected']}")
    pred = generate_sql(model, tokenizer, tc["schema"], tc["question"])
    print(f"Predicted: {pred}")
print("\n✅ Done — compare predicted vs expected above")
print("💡 More training data (500–2000 examples) closes the gap significantly.")

## 7. LLM-as-Judge Evaluation (L17 Pattern)

Exact SQL matching is too strict — there are multiple valid SQLs for the same query.

Instead, we use an **LLM judge** (Haiku — fast and cheap) that checks:
- **Correctness** (0–3): Does the SQL answer the question correctly?
- **Syntax** (0–2): Is it valid SQL?
- **Schema adherence** (0–2): Does it use the right tables/columns?
- **Efficiency** (0–1): Is it reasonably efficient?

This is the same evaluation pattern from Lesson 17 (Advanced Evals), now applied to code generation.

The judge scores each prediction and gives a `correct | partial | incorrect` verdict.

In [ ]:
client = anthropic.Anthropic()

def judge_sql(schema: str, question: str, predicted: str, reference: str) -> dict:
    prompt = f"""You are a SQL evaluation expert.

Schema: {schema}
Question: {question}
Reference SQL (correct): {reference}
Predicted SQL: {predicted}

Score the predicted SQL:
1. CORRECTNESS (0-3): correctly answers the question?
2. SYNTAX (0-2): valid SQL?
3. SCHEMA_ADHERENCE (0-2): uses correct tables/columns?
4. EFFICIENCY (0-1): reasonably efficient?

Respond ONLY with this JSON (no code fences):
{{"correctness": <0-3>, "syntax": <0-2>, "schema_adherence": <0-2>, "efficiency": <0-1>, "total": <0-8>, "verdict": "correct|partial|incorrect", "reason": "<one sentence>"}}"""

    resp = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    text = resp.content[0].text.strip()
    # Strip code fences if present
    if "```" in text:
        lines = text.split("\n")
        text = "\n".join(l for l in lines if not l.startswith("```"))
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"total": 0, "verdict": "parse_error", "reason": text[:80]}

# ── Evaluate fine-tuned model on test cases ───────────────────────────────────
print("Running LLM-judge evaluation (claude-haiku-4-5) ...\n")
results = []
for i, tc in enumerate(test_cases):
    pred = generate_sql(model, tokenizer, tc["schema"], tc["question"])
    score = judge_sql(tc["schema"], tc["question"], pred, tc["expected"])
    results.append(score)
    verdict = score.get("verdict", "?").upper()
    total   = score.get("total", 0)
    reason  = score.get("reason", "")
    print(f"Test {i+1}: {verdict:10s} score={total}/8  — {reason}")

avg = sum(r.get("total", 0) for r in results) / len(results)
print(f"\n{'─'*50}")
print(f"Average score: {avg:.1f} / 8")
print(f"\n💡 With 1K training examples, typical scores reach 6–7/8.")
print(f"   The judge harness is reusable — swap in any model for comparison.")

## 8. Save the Adapter, Merge, and Serve with vLLM

### Three Deployment Options

**Option A — Load base + adapter at runtime**
```python
base  = AutoModelForCausalLM.from_pretrained(base_name, quantization_config=bnb_config)
model = PeftModel.from_pretrained(base, adapter_path)
```
✅ Tiny artifact (adapter < 100 MB)
❌ Slightly slower inference (adapter computation on every forward pass)

**Option B — Merge adapter into base (W_final = W_base + A×B)**
```python
merged = model.merge_and_unload()      # fuses, removes PEFT wrapper
merged.save_pretrained(merged_path)
```
✅ Fastest inference — zero adapter overhead
✅ Drop-in for vLLM (load like any normal HuggingFace model)
❌ Larger disk footprint (full model size)

**Option C — Push merged model to HuggingFace Hub**
```python
merged.push_to_hub("username/qwen-sql-finetuned")
```
✅ Shareable, versionable, directly loadable by vLLM in production

**We do Option B below, then wire into vLLM.**

In [ ]:
# ── Merge adapter into base model ────────────────────────────────────────────
MERGED_DIR = "/content/qlora_sql_merged"

print("Merging LoRA adapter into base weights ...")
print("merge_and_unload() computes: W_final = W_base + A @ B for each adapted layer")
t0 = time.time()
merged_model = model.merge_and_unload()
print(f"✅ Merge complete in {time.time()-t0:.1f}s")
print(f"   Footprint after merge: {merged_model.get_memory_footprint()/1e9:.2f} GB")

print(f"\nSaving to {MERGED_DIR} (safe_serialization=True for vLLM compatibility) ...")
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

# Show saved files
print("\nSaved files:")
for f in sorted(os.listdir(MERGED_DIR)):
    sz = os.path.getsize(f"{MERGED_DIR}/{f}")
    print(f"  {f:<45} {sz/1e6:>7.1f} MB")

import subprocess
du = subprocess.run(["du", "-sh", MERGED_DIR], capture_output=True, text=True).stdout.strip()
print(f"\nTotal: {du}")
print("\n✅ Merged model ready for vLLM serving")

## 9. Serve the Fine-Tuned Model with vLLM (L37 Bridge)

This is the payoff of combining L37 + L38:

| What you learned | Where it applies |
|---|---|
| L37: vLLM PagedAttention + continuous batching | High-throughput inference engine |
| L38: QLoRA fine-tuning + adapter merging | Custom model that beats general models on your task |
| **Combined** | **Serve YOUR fine-tuned model at production throughput** |

The vLLM command is **identical to L37** — only the `--model` path changes:

```bash
# L37 — serving a base model:
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --dtype bfloat16 --max-model-len 2048 --port 8000

# L38 — serving YOUR fine-tuned merged model:
python -m vllm.entrypoints.openai.api_server \
    --model /content/qlora_sql_merged \
    --dtype bfloat16 --max-model-len 2048 --port 8000
```

The OpenAI-compatible client code from L37 works **unchanged** — the API is identical.
This is the architecture pattern: fine-tune once, serve with production infrastructure.

In [ ]:
# ── Reload merged model for inference test (clears VRAM from training) ────────
# In production you would start a fresh vLLM server process (see markdown above).
# In Colab we reload into transformers to stay within VRAM budget.

print("Clearing training model from VRAM ...")
del model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after clear: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

print(f"\nLoading merged model from {MERGED_DIR} ...")
t0 = time.time()
inf_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
inf_tok = AutoTokenizer.from_pretrained(MERGED_DIR)
print(f"✅ Loaded in {time.time()-t0:.1f}s  |  {inf_model.get_memory_footprint()/1e9:.2f} GB")

# ── Final inference test on merged model ──────────────────────────────────────
test_schema   = "CREATE TABLE sales (sale_id INT, product VARCHAR, amount DECIMAL, date DATE, region VARCHAR)"
test_question = "What is the total sales amount per region for the current year?"

print(f"\nTest inference on merged model:")
print(f"Q: {test_question}")
ans = generate_sql(inf_model, inf_tok, test_schema, test_question)
print(f"A: {ans}")

# ── Show the vLLM production command ─────────────────────────────────────────
print()
print("─" * 60)
print("Production vLLM serving command:")
print(f"""
  python -m vllm.entrypoints.openai.api_server \\
    --model {MERGED_DIR} \\
    --dtype bfloat16 \\
    --max-model-len 2048 \\
    --port 8000

  # Then query identically to L37:
  from openai import OpenAI
  client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
  resp = client.chat.completions.create(
      model="{MERGED_DIR}",
      messages=[{{"role": "user", "content": "Your SQL question..."}}]
  )
""")

## 10. Common QLoRA Pitfalls

| Pitfall | What Happens | Fix |
|---------|-------------|-----|
| **Eval on train data** | Metrics look great; production fails | Hold out test split BEFORE training. Never peek. |
| **LR too high** | Loss spikes, NaN gradients | Start at `2e-4` for LoRA. `1e-5` for full fine-tuning. |
| **Skipping warmup** | LR jump at step 0 destabilizes adapters | `warmup_ratio=0.05` minimum |
| **No gradient accumulation on small batch** | Noisy gradients, poor convergence | Effective batch ≥ 8 |
| **Packing disabled** | 50% of context window is padding | `packing=True` doubles throughput |
| **Rank too high on small dataset** | Overfitting — memorizes, doesn't generalize | `r=8` or `r=16` for < 1K examples |
| **Wrong target modules** | Adapters on wrong layers, no improvement | Adapt all `*_proj` layers in attention + FFN |
| **NF4 on CPU** | `bitsandbytes` requires CUDA | GPU required for quantized loading |
| **Chat template mismatch** | Gibberish output at inference | Always use `apply_chat_template` for both train AND inference |
| **Catastrophic forgetting** | Loses general capability | Limit epochs (3–5), use dropout, don't over-narrow the task |
| **Noisy labels** | 5–10% wrong labels significantly degrades quality | Audit data before training |

### When Fine-Tuning Doesn't Help
- **General knowledge**: Model already knows the facts; fine-tuning just noisily memorizes them
- **Rapidly changing data**: Fine-tuning bakes in stale data — use RAG (L7) for real-time info
- **< 50 examples**: Can't generalize — use few-shot prompting
- **Inconsistent labels**: Garbage in, garbage out

In [ ]:
# ── Cost math: fine-tuned 1.5B vs API alternatives ───────────────────────────
@dataclass
class CostScenario:
    name: str
    cost_per_1m_in:  float
    cost_per_1m_out: float
    setup_usd: float          # one-time training cost

    def monthly(self, volume_m=100) -> float:
        # assume 70% input, 30% output tokens
        return volume_m * (0.7 * self.cost_per_1m_in + 0.3 * self.cost_per_1m_out)

    def total_3m(self, volume_m=100) -> float:
        return self.monthly(volume_m) * 3 + self.setup_usd

scenarios = [
    CostScenario("Claude Haiku (API)",           0.80,  4.00,  0),
    CostScenario("Claude Sonnet (API)",           3.00, 15.00,  0),
    CostScenario("Self-hosted 7B vLLM (L37)",    0.012, 0.012,  0),
    CostScenario("Fine-tuned 1.5B vLLM (L38)",  0.005, 0.005,  5),
    CostScenario("Fine-tuned 0.5B vLLM",         0.002, 0.002,  3),
]

VOLUME_M = 100   # 100M tokens/month

print(f"Cost comparison at {VOLUME_M}M tokens/month for a narrow SQL task:")
print(f"{'Approach':<35} {'Monthly':>10} {'3-Month':>10} {'vs Haiku':>10}")
print("─" * 70)
haiku_3m = scenarios[0].total_3m(VOLUME_M)
for s in scenarios:
    m3 = s.total_3m(VOLUME_M)
    savings_pct = (haiku_3m - m3) / haiku_3m * 100
    print(f"{s.name:<35} ${s.monthly(VOLUME_M):>8,.0f} ${m3:>9,.0f} {savings_pct:>9.0f}%")

print()
print("Key takeaway:")
print("  Fine-tuned 1.5B saves ~99% of Haiku API costs for narrow tasks at 100M tok/month.")
print(f"  The $5 QLoRA training cost pays back in the first hour at this volume.")

## 11. Homework

1. **Scale up the data**: Change `TRAIN_SIZE = 500`. Compare judge scores. At what data size does quality plateau?

2. **Change the task**: Fine-tune on a different dataset:
   - `iamtarun/python_code_instructions_18k_alpaca` (Python code generation)
   - `philschmid/sql-create-context-copy` (alternative SQL set)
   - Build your own 100-example dataset for your specific use case

3. **Experiment with rank**: Try `r = 8`, `r = 32`, `r = 64` on the same data. Plot training loss curves. What's the expressiveness vs overfitting tradeoff?

4. **Upload to HuggingFace Hub**: After merging, publish your model:
   ```python
   merged_model.push_to_hub("your-username/qwen-sql-qlora")
   tokenizer.push_to_hub("your-username/qwen-sql-qlora")
   ```
   Then serve from the Hub with vLLM: `--model your-username/qwen-sql-qlora`

5. **Wire into the AutoResearcher (L36)**: Replace the Haiku "worker bee" in the Research Swarm's Searcher agent with your fine-tuned model for structured data retrieval. Measure cost savings and quality delta.

---

## Coming Next: Lesson 39 — DPO/ORPO: Preference Tuning

QLoRA (SFT) taught the model **what to say** by imitating good examples.
**DPO/ORPO** teaches it **which answer is better** from preference pairs.

```
SFT:  (prompt, good_response)
DPO:  (prompt, chosen_response, rejected_response)  ← prefers chosen over rejected
```

This is how modern "aligned" models are built (Llama-2-chat, Mistral-Instruct, Qwen-Instruct).
DPO (Rafailov et al. 2023) eliminates the need for a separate reward model — the policy itself is the reward.

In L39 you'll use `trl.DPOTrainer` to fine-tune for:
- Concise over verbose answers
- Structured (JSON) over free-form output
- Safe over unsafe completions

And you'll measure the quality delta: SFT alone vs SFT + DPO.
